Motif scanning with the full JASPAR database first and then filter for those expressed in myeloma is recommended than filter the JASPAR database for TF expressed in myeloma and then motif scanning

# Import pakcages

In [1]:
import re
import os
import pandas as pd
import plotly.express as px

# Custom functions

In [2]:
from tools.obtain_gene_seq import gene_name_to_download_seq
from tools.fimo_motif_scanning import motif_scanning, filter_fimo_output_by_tf_list, merge_rank_by_count_best_q

# Obtain sequence with interested region

In [3]:
result_dict = {
    'NT5C2': {
        'cell_line': 'ac' #Filtered in astrocytes cell lines
    },
    'NEGR1': {
        'cell_line': 'mbm' #Filtered in MBM cell lines
    },
    'RFTN1': {
        'cell_line': 'um' #Filted in UM cell lines
    }
}

In [4]:
for gene in result_dict.keys():
    seq_direct = gene_name_to_download_seq(
        gene_name = gene,
        seq_type ='promoter',
        upstream_bp = 5000,
        downstream_bp = 2000,
        save_file = True
    )

    result_dict[gene]['seq_direct'] = seq_direct

Sequence is already available at input/obtained_seq/seq_NT5C2_promoter_5000_2000.fna
Sequence is already available at input/obtained_seq/seq_NEGR1_promoter_5000_2000.fna
Sequence is already available at input/obtained_seq/seq_RFTN1_promoter_5000_2000.fna


# fimo motif scanning

In [5]:
! fimo --version # Run this line to ensure your interpreter can find the finmo software

/bin/bash: warning: setlocale: LC_ALL: cannot change locale (en_US.UTF-8)
5.5.8


In [6]:
p_threshold = 1E-4
motif_database_direct = 'input/jaspar_dataset/JASPAR2026_CORE_vertebrates_non-redundant_pfms_meme_original.txt'  # This is the motif database you downloaded from JASPAR

for gene in result_dict.keys():
    direct_fimo_output = f'output/{gene}_jaspar_human_full/fimo_output'
    direct_seq_input = f'input/obtained_seq/seq_{gene}_promoter_5000_2000.fna'

    os.makedirs(direct_fimo_output, exist_ok=True)

    result_direct = motif_scanning(
        direct_fimo_output,
        motif_database_direct,
        direct_seq_input,
        p_threshold
    )

    result_dict[gene]['fimo_output_direct'] = result_direct

Running command: fimo --oc output/NT5C2_jaspar_human_full/fimo_output --verbosity 1 --bgfile --nrdb-- --thresh 0.0001 --no-pgc input/jaspar_dataset/JASPAR2026_CORE_vertebrates_non-redundant_pfms_meme_original.txt input/obtained_seq/seq_NT5C2_promoter_5000_2000.fna
Motif scanning result saved as output/NT5C2_jaspar_human_full/fimo_output/fimo.tsv
Running command: fimo --oc output/NEGR1_jaspar_human_full/fimo_output --verbosity 1 --bgfile --nrdb-- --thresh 0.0001 --no-pgc input/jaspar_dataset/JASPAR2026_CORE_vertebrates_non-redundant_pfms_meme_original.txt input/obtained_seq/seq_NEGR1_promoter_5000_2000.fna
Motif scanning result saved as output/NEGR1_jaspar_human_full/fimo_output/fimo.tsv
Running command: fimo --oc output/RFTN1_jaspar_human_full/fimo_output --verbosity 1 --bgfile --nrdb-- --thresh 0.0001 --no-pgc input/jaspar_dataset/JASPAR2026_CORE_vertebrates_non-redundant_pfms_meme_original.txt input/obtained_seq/seq_RFTN1_promoter_5000_2000.fna
Motif scanning result saved as output/R

# Result analyses

## Read in result file

In [7]:
for gene in result_dict.keys():
    df_gene = pd.read_csv(result_dict[gene]['fimo_output_direct'], sep='\t')

    df_gene['TF'] = df_gene['motif_alt_id'].fillna(df_gene['motif_id'])

    df_gene['TF_stripped'] = df_gene['TF'].apply(
        lambda name: re.sub(r'[^A-Za-z0-9]', '', name))  #Create a new col for stripped TF name

    result_dict[gene]['df'] = df_gene

## Filter each df by their TF

In [8]:
for gene in result_dict.keys():
    print(f'>>>> Processing fimo output for {gene}')

    tf_pkl_direct = f'input/filters/tf_{result_dict[gene]['cell_line']}.pkl'
    fimo_df = result_dict[gene]['df']

    fimo_df_filtered = filter_fimo_output_by_tf_list(tf_pkl_direct, fimo_df)

    result_dict[gene]['df_filtered'] = fimo_df_filtered

>>>> Processing fimo output for NT5C2
Before filtering, df has a shape as (2446, 12)
After filtering, df has a shape as (1561, 12)
>>>> Processing fimo output for NEGR1
Before filtering, df has a shape as (2224, 12)
After filtering, df has a shape as (1334, 12)
>>>> Processing fimo output for RFTN1
Before filtering, df has a shape as (2514, 12)
After filtering, df has a shape as (1415, 12)


## Generate & save merged dataset

In [9]:
for gene in result_dict.keys():
    df_save_direct = f'output/{gene}_jaspar_human_full/{gene}_{result_dict[gene]['cell_line']}_filter_hit_tf_wise.csv'
    merged_df = merge_rank_by_count_best_q(result_dict[gene]['df_filtered'], df_save_direct)
    result_dict[gene]['merged_df'] = merged_df

## Plot binding quality vs. number of binding sites

In [10]:
for gene in result_dict.keys():
    cell_line_capitalized = result_dict[gene]['cell_line'].upper()

    fig = px.scatter(
        result_dict[gene]['merged_df'],
        x="site_count",
        y="best_q_per_tf",
        hover_name="TF",  # big bold line in tooltip
        hover_data={
            "site_count": True,  # show extra fields in tooltip
            "best_q_per_tf": True,
            "TF": False
        },  # don't duplicate TF in the table
        labels={
            "site_count": "Sites per TF",
            "best_q_per_tf": "Best q score per TF",
        },
        title=f'{gene} in {cell_line_capitalized} cell lines: TF binding quality vs. binding site count',
        width=800,
        height=800
    )

    # nicer layout + grid
    fig.update_layout(template="plotly_white")

    fig.show()
    fig.write_html(f'output/{gene}_jaspar_human_full/{gene}_{cell_line_capitalized}_filter_hit_tf.html')